# GSM8K 深度–正确率评测（Qwen3-4B）

**目标**：用 Qwen3-4B 在 GSM8K socratic 测试集（1319 题）上测准确率，并按「题目深度」分组统计正确率。

**深度口径（本 notebook 的定义）**
GSM8K 的 socratic 版 gold solution 会把每一步计算写成 `<<表达式=结果>>`。本 notebook 定义：

> **depth = 该题 gold solution 中 `<<...>>` 的个数**，即一次计算 +1。
>
> 注意：`<<16-3-4=9>>` 内部即使有两个运算符，也只算 1 步；本口径按注解个数计，不拆原子运算。

这与仓库早期 `gsm8k_dataset/test_d.jsonl` 里的图链深度 `d` 不同：那个 `d` 要求前后注解按数值连边、再取最长链；本表按本次要求直接数注解，等价于早期的 `n_ann` / `W`。两者不要混用。

**产物（都在 `Experiments/gsm8k_results/`）**
- 逐题记录 JSONL：含 `depth / gold / pred / correct / output` 原文
- 按深度汇总 CSV
- 正确率–深度图 PNG
- 屏幕输出：总体正确率、按深度表、累计表、错误样例

**运行建议**
1. 正式结果需要 GPU（Colab / WSL）；Windows 原生不要跑 vLLM（见项目 `AGENTS.md`）。
2. 先设 `GSM8K_LIMIT=100` 试跑，确认流程与显存，再设 `GSM8K_LIMIT=None` 跑全量。
3. 无 GPU 时可用 `GSM8K_BACKEND=mock` 验证数据 / 深度 / 统计 / 绘图流程；**mock 的数字是假的，不能当结果**。
4. 中断后重跑同一配置会按 `uid` 自动续跑。

In [ ]:
# ===== 全局配置 =====
import os, re, sys, json, math, time, random, hashlib, importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# 项目规定的 4B 档位：Qwen3-4B-Instruct-2507（非 base、非 thinking）
# 换本地权重：export GSM8K_MODEL=models/Qwen3-4B-Instruct-2507
MODEL_NAME = os.environ.get("GSM8K_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
SPLIT = os.environ.get("GSM8K_SPLIT", "test")

# 运行规模：先小样本试跑，确认后再设 None 跑全量
_limit = os.environ.get("GSM8K_LIMIT", "")
LIMIT = int(_limit) if _limit not in ("", "None", "none") else None

# 后端：auto / vllm / transformers / mock
BACKEND = os.environ.get("GSM8K_BACKEND", "auto")

# 采样参数：温度 / top_p 沿用项目统一值；GSM8K 输出较短，max_tokens 取 2048
TEMPERATURE = float(os.environ.get("GSM8K_TEMPERATURE", "0.6"))
TOP_P = 0.95
MAX_NEW_TOKENS = int(os.environ.get("GSM8K_MAX_NEW_TOKENS", "2048"))
BATCH_SIZE = int(os.environ.get("GSM8K_BATCH_SIZE", "8"))
SEED = int(os.environ.get("GSM8K_SEED", "0"))


def find_repo_root(start=None):
    """从当前目录向上找仓库根（含 gsm8k_dataset/ 或 pyproject.toml）。"""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "gsm8k_dataset").exists() or (cand / "pyproject.toml").exists():
            return cand
    return p


ROOT = find_repo_root()
DATA_DIR = ROOT / "gsm8k_dataset"
RESULTS_DIR = ROOT / "Experiments" / "gsm8k_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 本地已下载的 4B 权重优先，否则走 HF repo id
if Path(MODEL_NAME).expanduser().exists():
    MODEL_PATH = str(Path(MODEL_NAME).expanduser())
else:
    _local = ROOT / "models" / "Qwen3-4B-Instruct-2507"
    MODEL_PATH = str(_local) if _local.exists() else MODEL_NAME

print("仓库根目录 :", ROOT)
print("Python     :", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
print("模型       :", MODEL_PATH)
print("数据/结果  :", DATA_DIR, "/", RESULTS_DIR)
print("规模       :", "全量" if LIMIT is None else f"前 {LIMIT} 题",
      f"| batch={BATCH_SIZE} | max_new_tokens={MAX_NEW_TOKENS}")

## 1. 数据与深度

本地 `gsm8k_dataset/test.jsonl` 是 socratic 版（由早期 notebook 从 `openai/gsm8k` 的 `socratic` 配置导出）；不存在时自动下载并落盘。

深度直接由 gold answer 里的 `<<...>>` 数量得到，**不依赖模型输出**。

In [ ]:
# ===== 读取 GSM8K socratic 测试集，并按 <<...>> 计算深度 =====
ANN_RE = re.compile(r"<<[^<>]*>>")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_gsm8k(split="test"):
    local = DATA_DIR / f"{split}.jsonl"
    if local.exists():
        return pd.DataFrame(read_jsonl(local)), f"local socratic ({local.name})"
    # 本地没有就下载 socratic 版，并落盘
    ds = load_dataset("openai/gsm8k", "socratic", split=split)
    df = ds.to_pandas()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_json(local, orient="records", lines=True, force_ascii=False)
    return df, "huggingface openai/gsm8k[socratic]"


raw_df, source = load_gsm8k(SPLIT)
df = raw_df.reset_index(drop=True)
df["uid"] = [f"{SPLIT}-{i:04d}" for i in range(len(df))]
df["depth"] = df["answer"].map(lambda s: len(ANN_RE.findall(str(s))))
# 与直接数 "<<" / ">>" 交叉验证，防止正则漏匹配
assert (df["depth"] == df["answer"].str.count(r"<<")).all(), "深度计数与 '<<' 数量不一致"
assert (df["depth"] == df["answer"].str.count(r">>")).all(), "深度计数与 '>>' 数量不一致"

print(f"数据来源: {source} | 题数: {len(df)}")
print(df[["question", "depth"]].head(3).to_string(index=False, max_colwidth=60))

dist = (df["depth"].value_counts().sort_index()
          .rename_axis("深度").reset_index(name="题数"))
dist["占比"] = dist["题数"] / len(df)
print("\n全量测试集的深度分布：")
display(dist.style.format({"占比": "{:.1%}"}))

ex = df.iloc[0]
print(f"\n示例：depth={ex['depth']}（数一数下面 answer 里的 <<>>）")
print(ex["answer"])

# 与仓库早期 test_d.jsonl 交叉核对（该文件非必需）
old_path = DATA_DIR / "test_d.jsonl"
if old_path.exists():
    old = pd.DataFrame(read_jsonl(old_path))
    if "n_ann" in old and len(old) == len(df):
        print("\n与 test_d.jsonl 的 n_ann 完全一致:",
              bool((old["n_ann"] == df["depth"]).all()))
        print("注：test_d.jsonl 的 d 是图链深度（数值连边最长链），不是本表的注解数。")

## 2. Prompt 与答案解析

- system prompt 要求模型以 `#### <number>` 结尾（沿用项目里 Qwen3-4B 测试的格式）。
- gold 取 `####` 后的数字；预测优先取最后一个 `####`，其次 `\boxed{}`，最后取全文最后一个数字。
- 若 chat template 含 `enable_thinking`（base / thinking 版 Qwen3），自动传 `enable_thinking=False`；`Qwen3-4B-Instruct-2507` 没有这个开关，不会传。

In [ ]:
# ===== Prompt 与答案抽取 =====
SYSTEM_PROMPT = """You are a math problem solver. Solve the problem carefully step by step.

The final line of your response MUST be exactly in this format:
#### <numeric answer>

where <numeric answer> is a single number (no units, no commas, no extra words). Do not write anything after the #### line."""


def build_messages(question):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]


NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def parse_number(text):
    """从一段文字里取最后一个数字；支持 $、逗号、百分号、Unicode 负号。"""
    if text is None:
        return None
    s = str(text).replace("−", "-").replace("$", "").replace("，", ",")
    nums = NUM_RE.findall(s)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", ""))
    except ValueError:
        return None


def parse_gold(answer):
    """GSM8K gold：#### 后面的数字。"""
    s = str(answer)
    if "####" in s:
        return parse_number(s.rsplit("####", 1)[1])
    return parse_number(s.strip().splitlines()[-1])


def parse_pred(text):
    """模型输出：优先取最后一个 #### 行，其次 \\boxed{}，最后取全文最后一个数。"""
    text = str(text).replace("<|im_end|>", "")
    if "####" in text:
        return parse_number(text.rsplit("####", 1)[1])
    m = re.search(r"\\boxed\{([^}]*)\}", text)
    if m:
        return parse_number(m.group(1))
    return parse_number(text)


def is_correct(pred, gold, tol=1e-4):
    return pred is not None and gold is not None and abs(pred - gold) <= tol


df["gold"] = df["answer"].map(parse_gold)
print("gold 解析失败:", int(df["gold"].isna().sum()))
assert df["gold"].notna().all(), "存在无法解析的 gold 答案，请检查数据"

# 小单元测试
assert parse_gold("... \n#### 18") == 18
assert parse_pred("blah\n#### 72") == 72
assert parse_pred("The answer is $1,000.") == 1000
assert parse_pred("#### -3") == -3
assert is_correct(9.0, 9) and not is_correct(9.1, 9)
print("答案解析单元测试通过")

## 3. 推理后端

- `vllm`：GPU 推荐。按项目 `AGENTS.md` 的模板，在 import 前设置 `VLLM_USE_FLASHINFER_SAMPLER=0`；T4（compute capability < 8）自动用 `dtype=half` + `enforce_eager=True`，更好的卡用 bf16。
- `transformers`：CPU / 无 vLLM 时兜底，速度慢很多。
- `mock`：不加载模型，只验证后续统计与绘图流程（**假结果**）。

In [ ]:
# ===== 推理后端：vLLM / transformers / mock =====
def resolve_backend_kind(requested):
    if requested != "auto":
        return requested
    if torch.cuda.is_available() and importlib.util.find_spec("vllm") is not None:
        return "vllm"
    return "transformers"


KIND = resolve_backend_kind(BACKEND)


class ChatBackend:
    """统一封装三种后端，接口：generate(rows) -> [{text, n_tokens, finish_reason}]"""

    def __init__(self, kind):
        self.kind = kind
        self.model_path = MODEL_PATH
        self.chat_kwargs = {}
        self._rng = random.Random(SEED)
        if kind == "vllm":
            self._init_vllm()
        elif kind == "transformers":
            self._init_transformers()
        elif kind == "mock":
            print("=" * 78)
            print("警告：MOCK 后端不加载模型，只随机伪造答案，用于验证数据/深度/统计/绘图流程。")
            print("      这里的正确率没有任何实验意义，不能写进报告。")
            print("=" * 78)
        else:
            raise ValueError(f"未知后端: {kind}")

    def _thinking_kwargs(self, tokenizer):
        template = getattr(tokenizer, "chat_template", "") or ""
        return {"enable_thinking": False} if "enable_thinking" in template else {}

    def _init_vllm(self):
        # vLLM 必须在 import 前关闭 FlashInfer 采样器（无 nvcc 时避免 JIT 失败）
        os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
        from vllm import LLM, SamplingParams

        major, _ = torch.cuda.get_device_capability(0)
        dtype = os.environ.get("GSM8K_VLLM_DTYPE") or ("half" if major < 8 else "bfloat16")
        self.sp = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P,
                                 max_tokens=MAX_NEW_TOKENS, seed=SEED)
        self.llm = LLM(
            model=self.model_path,
            dtype=dtype,
            gpu_memory_utilization=float(os.environ.get("GSM8K_GPU_MEM", "0.80")),
            max_model_len=int(os.environ.get("GSM8K_MAX_MODEL_LEN", "4096")),
            max_num_seqs=int(os.environ.get("GSM8K_MAX_NUM_SEQS", "8")),
            enforce_eager=(major < 8),  # T4 等 Turing 卡走 eager，避免后端兼容问题
        )
        self.tok = self.llm.get_tokenizer()
        self.chat_kwargs = self._thinking_kwargs(self.tok)
        print(f"vLLM 已加载: dtype={dtype} | max_model_len={os.environ.get('GSM8K_MAX_MODEL_LEN', '4096')}")

    def _init_transformers(self):
        self.tok = AutoTokenizer.from_pretrained(self.model_path, trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.tok.padding_side = "left"
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path, dtype="auto", device_map="auto")
        except TypeError:  # 旧版 transformers 用 torch_dtype
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path, torch_dtype="auto", device_map="auto")
        self.model.eval()
        self.chat_kwargs = self._thinking_kwargs(self.tok)
        print(f"transformers 已加载: device={self.model.device} | dtype={self.model.dtype}")

    @torch.inference_mode()
    def generate(self, rows):
        if self.kind == "mock":
            out = []
            for r in rows:
                gold = r.get("gold")
                # 80% 概率答对，仅让表格有非平凡形态；这是假数据
                if gold is not None and self._rng.random() < 0.80:
                    text = f"#### {gold:g}"
                else:
                    text = f"#### {(gold or 0) + 1:g}"
                out.append({"text": text, "n_tokens": 8, "finish_reason": "stop"})
            return out

        messages = [build_messages(r["question"]) for r in rows]

        if self.kind == "vllm":
            kwargs = {"chat_template_kwargs": self.chat_kwargs} if self.chat_kwargs else {}
            outs = self.llm.chat(messages, self.sp, **kwargs)
            return [{"text": o.outputs[0].text,
                     "n_tokens": len(o.outputs[0].token_ids),
                     "finish_reason": o.outputs[0].finish_reason} for o in outs]

        # transformers：一次 batch 生成，左 padding
        texts = [self.tok.apply_chat_template(
                    m, tokenize=False, add_generation_prompt=True, **self.chat_kwargs)
                 for m in messages]
        max_input = max(256, int(os.environ.get("GSM8K_MAX_MODEL_LEN", "4096")) - MAX_NEW_TOKENS)
        enc = self.tok(texts, return_tensors="pt", padding=True,
                       truncation=True, max_length=max_input).to(self.model.device)
        gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS,
                          pad_token_id=self.tok.pad_token_id)
        if TEMPERATURE > 0:
            gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
        else:
            gen_kwargs.update(do_sample=False)
        gen = self.model.generate(**enc, **gen_kwargs)

        out = []
        prompt_len = enc["input_ids"].shape[1]
        for i in range(len(rows)):
            ids = gen[i][prompt_len:]
            stop = self.tok.eos_token_id in ids.tolist()
            out.append({"text": self.tok.decode(ids, skip_special_tokens=True),
                        "n_tokens": len(ids),
                        "finish_reason": "stop" if stop else "length"})
        return out

## 4. 逐题推理

结果按 batch 追加写入 JSONL；同一 `fingerprint`（模型、prompt、采样参数、深度口径）下按 `uid` 续跑。

In [ ]:
# ===== 逐题推理（带断点续跑）=====
def sha8(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:8]


def model_slug(name):
    # 本地路径取目录名；HF repo id 取最后一段
    s = re.sub(r"[^0-9A-Za-z]+", "-", Path(str(name)).name).strip("-").lower()
    return s[-60:] if len(s) > 60 else s


FINGERPRINT = sha8({
    "model": MODEL_PATH,
    "prompt": SYSTEM_PROMPT,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
    "seed": SEED,
    "depth_def": "count(<<...>>)",
    "data": sha8({"n": len(df), "questions": df["question"].tolist()}),
})
STEM = f"{model_slug(MODEL_PATH)}_{SPLIT}_{FINGERPRINT}"
RAW_PATH = RESULTS_DIR / f"{STEM}.jsonl"
SUMMARY_PATH = RESULTS_DIR / f"{STEM}_by_depth.csv"
PER_Q_PATH = RESULTS_DIR / f"{STEM}_per_question.csv"
PLOT_PATH = RESULTS_DIR / f"{STEM}_accuracy_by_depth.png"
print("结果文件:", RAW_PATH)


def load_cached(path, fp):
    if not path.exists():
        return []
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            if r.get("fp") == fp:
                recs.append(r)
    return recs


def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def run_eval(limit=None):
    global KIND
    todo = (df if limit is None else df.head(limit)).to_dict("records")
    todo_ids = {r["uid"] for r in todo}
    cached = [r for r in load_cached(RAW_PATH, FINGERPRINT) if r["uid"] in todo_ids]
    done = {r["uid"] for r in cached}
    pending = [r for r in todo if r["uid"] not in done]
    print(f"计划 {len(todo)} 题 | 已完成 {len(done)} | 待跑 {len(pending)} | 后端 {KIND}")

    if not pending:
        print("缓存已覆盖全部题目，直接汇总。")
        return pd.DataFrame(cached)

    try:
        backend = ChatBackend(KIND)
    except Exception as e:
        if KIND == "vllm":
            print("vLLM 启动失败，回退 transformers：", repr(e))
            KIND = "transformers"
            backend = ChatBackend(KIND)
        else:
            raise

    t_start = time.time()
    for offset in range(0, len(pending), BATCH_SIZE):
        batch = pending[offset:offset + BATCH_SIZE]
        t0 = time.time()
        gens = backend.generate(batch)
        dt = time.time() - t0
        recs = []
        for row, g in zip(batch, gens):
            pred = parse_pred(g["text"])
            recs.append({
                "fp": FINGERPRINT,
                "model": MODEL_PATH,
                "backend": KIND,
                "uid": row["uid"],
                "split": SPLIT,
                "question": row["question"],
                "gold": row["gold"],
                "pred": pred,
                "correct": bool(is_correct(pred, row["gold"])),
                "depth": int(row["depth"]),
                "n_tokens": int(g["n_tokens"]),
                "finish_reason": g["finish_reason"],
                "truncated": g["finish_reason"] == "length",
                "seconds": round(dt / len(batch), 2),
                "output": g["text"],
            })
        append_jsonl(RAW_PATH, recs)

        finished = len(done) + offset + len(batch)
        batch_acc = sum(r["correct"] for r in recs) / len(recs)
        remain = len(pending) - offset - len(batch)
        eta = (time.time() - t_start) / (offset + len(batch)) * remain
        print(f"[{finished}/{len(todo)}] batch_acc={batch_acc:.0%} "
              f"耗时={dt:.1f}s | 预计剩余 {eta / 60:.1f} min")

    recs = [r for r in load_cached(RAW_PATH, FINGERPRINT) if r["uid"] in todo_ids]
    print(f"\n完成，共 {len(recs)} 条记录；用时 {(time.time() - t_start) / 60:.1f} min")
    return pd.DataFrame(recs)


res = run_eval(LIMIT)
print(res[["uid", "depth", "gold", "pred", "correct"]].head().to_string(index=False))

## 5. 总体正确率与按深度分组

正确率 = 预测数值与 gold 数值一致（容差 `1e-4`）。
每个深度给出 Wilson 95% 置信区间；高深度样本少时区间会很宽。

In [ ]:
# ===== 总体正确率 + 按深度分组 =====
res = res.drop_duplicates("uid", keep="last").copy()
assert len(res) > 0, "没有可用记录"


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    center = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, center - half), min(1.0, center + half)


n_all = len(res)
k_all = int(res["correct"].sum())
acc_all = k_all / n_all
lo, hi = wilson_ci(k_all, n_all)
print(f"总体正确率: {acc_all:.2%} ({k_all}/{n_all})，"
      f"Wilson 95% CI [{lo:.2%}, {hi:.2%}]")
print(f"截断率: {res['truncated'].mean():.2%} | "
      f"解析失败率: {res['pred'].isna().mean():.2%}")

g = (res.groupby("depth")
        .agg(题数=("correct", "size"),
             正确数=("correct", "sum"),
             平均输出token=("n_tokens", "mean"),
             平均耗时秒=("seconds", "mean"))
        .reset_index()
        .rename(columns={"depth": "深度"}))
g["正确率"] = g["正确数"] / g["题数"]
cis = g.apply(lambda r: wilson_ci(r["正确数"], r["题数"]), axis=1)
g["CI下限"] = [x[0] for x in cis]
g["CI上限"] = [x[1] for x in cis]

print("\n按深度分组的正确率：")
display(g.style.format({"正确率": "{:.2%}", "CI下限": "{:.2%}", "CI上限": "{:.2%}",
                        "平均输出token": "{:.0f}", "平均耗时秒": "{:.1f}"}))
print(g.to_string(index=False))

# 累计：深度 <= d 的合并正确率
cum_rows = []
for d in sorted(g["深度"]):
    sub = res[res["depth"] <= d]
    l, h = wilson_ci(int(sub["correct"].sum()), len(sub))
    cum_rows.append({"深度上限": d, "题数": len(sub),
                     "正确数": int(sub["correct"].sum()),
                     "累计正确率": sub["correct"].mean(),
                     "CI下限": l, "CI上限": h})
cum = pd.DataFrame(cum_rows)

print("\n累计正确率（深度 <= d）：")
display(cum.style.format({"累计正确率": "{:.2%}", "CI下限": "{:.2%}", "CI上限": "{:.2%}"}))
print(cum.to_string(index=False))

g.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
res[["uid", "depth", "gold", "pred", "correct", "n_tokens",
     "finish_reason", "seconds"]].to_csv(
    PER_Q_PATH, index=False, encoding="utf-8-sig")
print("\n已保存:", SUMMARY_PATH)
print("已保存:", PER_Q_PATH)

## 6. 正确率–深度图

In [ ]:
# ===== 图：正确率随深度变化 =====
fig, ax = plt.subplots(figsize=(8, 4.2))
x = g["深度"].to_numpy()
y = g["正确率"].to_numpy()
err = np.vstack([y - g["CI下限"].to_numpy(), g["CI上限"].to_numpy() - y])
ax.bar(x, y, width=0.65, color="#4C78A8", alpha=0.85,
       yerr=err, capsize=3, error_kw={"elinewidth": 1})
ax.axhline(acc_all, color="#E45756", ls="--", lw=1.2,
           label=f"Overall {acc_all:.1%}")
for xi, yi, ni in zip(x, y, g["题数"]):
    ax.text(xi, min(yi + 0.03, 1.02), f"n={ni}", ha="center", fontsize=8)
ax.set_xlabel("Depth (number of <<>> calculation steps in gold solution)")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.08)
ax.set_xticks(x)
ax.set_title(f"{Path(MODEL_PATH).name} on GSM8K {SPLIT} (n={n_all})")
ax.legend(loc="best", frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("已保存:", PLOT_PATH)

## 7. 错误样例

用于人工检查错误是「算错」还是「格式 / 截断」。

In [ ]:
# ===== 错误样例（最多 8 条，按深度排序）=====
err = res[~res["correct"]].sort_values(["depth", "uid"]).head(8)
if err.empty:
    print("没有错误样例。")
else:
    for _, r in err.iterrows():
        print("=" * 90)
        print(f"[{r['uid']}] depth={r['depth']} gold={r['gold']} "
              f"pred={r['pred']} tokens={r['n_tokens']} finish={r['finish_reason']}")
        print("Q:", str(r["question"])[:300])
        tail = str(r["output"]).strip().splitlines()[-4:]
        print("A(tail):", "\n".join(tail)[:500])

## 8. 解读注意事项

1. **深度样本不平衡**：测试集大部分题在深度 2–5；深度 6 以上只有几十题，置信区间很宽，不要把小样本的波动当成趋势。
2. **depth 与工作量 / 题长相关**：深度越大的题通常计算步数和题干也更长，本表是条件正确率，不是「深度」的因果效应。要拆开需按同 `W` 配平（见项目 `AGENTS.md` 的 d×W 设计）。
3. **单次采样**：默认 `temperature=0.6`（项目统一采样），重跑会有波动；若要更稳，可把 `GSM8K_TEMPERATURE=0` 做贪心解码，或每题多次采样取均值。
4. **截断**：若 `截断率` 明显 > 0，先增大 `GSM8K_MAX_NEW_TOKENS`（同时注意 `GSM8K_MAX_MODEL_LEN`），否则会把「没写完」误算成答错。
5. **depth=0**：18 道题的 socratic gold 里没有 `<<>>`，单独一行报告；它们的正确率含义与其它深度略有不同。
6. **mock 结果**：如果日志里出现 MOCK 警告，说明没有真正加载模型，表里的数字只能用来检查代码流程。